# NumPy support in Numba

**Optional deep dive, about 15 minutes.**

Numba can compile loops that use NumPy arrays and functions that act on each array item. This notebook explores two extensions to the core lesson:

- How an array's number type and storage can make Numba compile another version.
- How `@vectorize` creates a function that acts on every array item.

As in the core notebook, check each result and call each new input type once before timing.

In [ ]:
import numpy as np
import numba
from numba import jit

print("NumPy", np.__version__)
print("Numba", numba.__version__)

## How Numba handles different array types

Numba can compile a new version when an input has a different number type, number of dimensions, or memory arrangement. Consider a function that changes small values to zero:

In [ ]:
@jit
def zero_clamp(x, threshold):
    # This function is designed for one-dimensional arrays.
    out = np.empty_like(x)
    for i in range(out.shape[0]):
        if np.abs(x[i]) > threshold:
            out[i] = x[i]
        else:
            out[i] = 0
    return out

In [ ]:
a_small = np.linspace(0, 1, 50)
zero_clamp(a_small, 0.3)

We will compare several arrays:

- `int64` with contiguous storage.
- `float32` with contiguous storage.
- `float32` with a stride, so elements are not contiguous in memory.

In [ ]:
n = 10000
a_int64 = np.arange(n, dtype=np.int64)
a_float32 = np.linspace(0, 1, n, dtype=np.float32)
a_float32_strided = np.linspace(0, 1, 2 * n, dtype=np.float32)[::2]

cases = (
    (a_int64, 1600),
    (a_float32, 0.3),
    (a_float32_strided, 0.3),
)

# Call and check every input type before timing it.
for values, threshold in cases:
    actual = zero_clamp(values, threshold)
    expected = np.where(np.abs(values) > threshold, values, 0)
    np.testing.assert_array_equal(actual, expected)

In [ ]:
%timeit -n 10 -r 3 zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 zero_clamp(a_float32_strided, 0.3)

The arrays contain the same number of items, but their number type and storage can change the timing. The result check above confirms that every compiled version still gives the right answer.

Compare with a clear NumPy version. Numba may avoid some extra work and temporary arrays, but timing both versions gives the answer:

In [ ]:
def np_zero_clamp(x, threshold):
    return np.where(np.abs(x) > threshold, x, 0)

In [ ]:
%timeit -n 10 -r 3 np_zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 np_zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 np_zero_clamp(a_float32_strided, 0.3)

## Create a function that acts on every array item

`@vectorize` tells Numba to apply one small function to every item in an array. NumPy already provides many fast array functions, so use this only when the function you need does not already exist.

In [ ]:
from numba import vectorize

In [ ]:
@vectorize
def ufunc_zero_clamp(x, threshold):
    if np.abs(x) > threshold:
        return x
    else:
        return 0

In [ ]:
for values, threshold in cases:
    actual = ufunc_zero_clamp(values, threshold)
    expected = np_zero_clamp(values, threshold)
    np.testing.assert_array_equal(actual, expected)

%timeit -n 10 -r 3 ufunc_zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 ufunc_zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 ufunc_zero_clamp(a_float32_strided, 0.3)

For this simple operation, the `@vectorize` version may be no faster than the compiled loop or NumPy. It is most useful for a custom operation that must be applied to every array item and is not already available in NumPy.

## Takeaway

- A new array number type or memory arrangement can make Numba compile another version.
- Check each result and call each input type once before timing.
- Prefer clear NumPy when it already provides the operation you need.